# 02. 수동 스킬 적용 경로 (사람이 통제)

`01`의 레거시 기준선을 사람이 통제하는 기본 경로로 개선하는 과정을 기록합니다.

```text
legacy-intake -> AIDM experiment -> AIDD promotion -> human review
```

각 단계는 이 저장소의 `.agents` 스킬에 대응합니다. 데모를 보는 사람은 자기 레거시(`01`)를 코딩 에이전트에게 맡겼을 때, 하네스가 **제안을 검증하고 게이트로 보호하며 사람 검토 지점에서 멈추는** 방식을 확인할 수 있습니다.

- 에이전트는 코드가 아니라 선언형 JSON 제안만 제출합니다.
- 승격 게이트(`decision`)를 우회하지 않습니다.
- AIDD는 검증·생성이며 배포가 아닙니다.

In [ ]:
from pathlib import Path

from power_forecasting.aidm import AIDMConfig
from power_forecasting.cli import run_aidm_workflow, run_aidd_workflow, run_generate_data

ARTIFACT_DIR = Path("artifacts/demo")
dataset_path = ARTIFACT_DIR / "dataset.csv"
if not dataset_path.exists():
    dataset_path = run_generate_data(ARTIFACT_DIR, days=60, plants=3, seed=42)

# legacy-intake / aidm-experiment: 에이전트가 제출하는 선언형 JSON 제안입니다.
# 예측 시점 입력(달력, 예보 물리량)만 사용하고 generation_mw / actual_* 는 쓰지 않습니다.
proposal = {
    "schema_version": "1",
    "proposal_id": "manual-demo-calendar-physics",
    "rationale": "Prediction-time calendar and solar-physics features with a bounded boosted-tree recipe.",
    "baseline": {"model": "SPOT"},
    "feature_sets": [
        {
            "name": "calendar_physics",
            "rationale": "Forecast-time solar physics plus daily calendar signals.",
            "specs": [
                {"name": "hour_sin", "transform": "cyclic_hour", "inputs": ["timestamp"], "parameters": {}, "rationale": "Daily cycle sine.", "version": "1"},
                {"name": "hour_cos", "transform": "cyclic_hour", "inputs": ["timestamp"], "parameters": {}, "rationale": "Daily cycle cosine.", "version": "1"},
                {"name": "effective_irradiance", "transform": "effective_irradiance", "inputs": ["forecast_irradiance", "forecast_cloud_cover"], "parameters": {}, "rationale": "Cloud-adjusted forecast irradiance.", "version": "1"},
            ],
        }
    ],
    "model_recipes": [
        {"name": "hgb_demo", "recipe": "hist_gradient_boosting", "parameters": {"max_iter": 200, "learning_rate": 0.1, "max_leaf_nodes": 31}, "rationale": "Bounded deterministic boosted trees."}
    ],
    "budget": {"max_evaluations": 4, "top_feature_groups": 2},
}

# 운영 기본 게이트: 최소 개선율 0.01, 발전소별 최대 저하 0.03.
config = AIDMConfig(folds=5, minimum_improvement=0.01, max_plant_regression=0.03, seed=42)
aidm_result = run_aidm_workflow(ARTIFACT_DIR, dataset=dataset_path, config=config, proposal=proposal)
manifest = aidm_result.manifest

print(f"결정(decision): {manifest['decision']}")
print(f"SPOT 기준선 NMAE: {manifest['baseline']['metrics']['nmae']:.6f}")
print(f"우승 후보: {manifest['winner']['name']}  NMAE: {manifest['winner']['metrics']['nmae']:.6f}")
print(f"개선율(improvement_ratio): {manifest['improvement_ratio']:.6f}")
print(f"실패한 게이트: {manifest['failed_gates']}")

In [ ]:
# aidd-promotion: 승격된 매니페스트만 결정론적 피처 모듈로 생성합니다.
if manifest["decision"] == "promote":
    manifest_path = ARTIFACT_DIR / "promotion_manifest.json"
    generated_module = run_aidd_workflow(ARTIFACT_DIR, manifest=manifest_path)
    print(f"생성된 모듈: {generated_module}")
    print("--- 앞부분 미리보기 ---")
    print("\n".join(generated_module.read_text(encoding="utf-8").splitlines()[:12]))
    print("\n[사람 검토 경계] 생성 모듈과 매니페스트는 검토 대상이며, 고객 시스템 배포는 사람 승인 이후에만 진행합니다.")
else:
    print("게이트가 승격을 거부했습니다. 하네스가 개선이 아닌 변경을 막았다는 뜻입니다.")
    print(f"실패한 게이트: {manifest['failed_gates']}")
    print("이 경우 AIDD를 실행하지 않으며, 제안을 수정해 다시 실험합니다.")

## 단계와 스킬 대응

| 단계 | 스킬 | 반드시 멈추는 경계 |
| --- | --- | --- |
| 레거시 연결·기준선 | `legacy-intake` | 사람 승인 전에는 실제 고객 시스템·데이터를 실행하지 않음 |
| 후보 탐색·게이트 비교 | `aidm-experiment` | `decision: reject`를 우회하지 않고 AIDD를 호출하지 않음 |
| 매니페스트 검증·모듈 생성 | `aidd-promotion` | 고객 저장소 수정·병합·배포를 하지 않음 |
| 종합 판정 | `release-gate` | 명시적 사람 승인 없이 release를 허용하지 않음 |

이 경로의 가치는 자동 개선 자체가 아니라, **개선을 주장하기 전에 검증 가능한 증적과 게이트를 남긴다는 점**입니다. `artifacts/demo/promotion_manifest.json`과 `generated/promoted_features.py`가 그 검토 증적입니다.